# YouTube Learning Assistant

This notebook turns a YouTube video transcript into a set of learning materials. It extracts the video ID, fetches and tokenizes the transcript, splits it into manageable chunks, and uses Gemini to create a combined summary. From that summary, it generates study notes and key concepts, then builds interactive flashcards and Mcq's.

Set `GOOGLE_API_KEY` in your environment before running the notebook. Run the cells in order.

### Import libraries for transcript processing, Gemini calls, token counting, and notebook output.


In [ ]:
import os
import re
import json

from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi
from openai import OpenAI

import tiktoken
import pandas as pd

from IPython.display import display, Markdown


### Load environment settings and initialize the Gemini client using `GOOGLE_API_KEY`.


In [ ]:
load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("No API Key was found.")

client = OpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=api_key
)

MODEL = "gemini-4-hydrogen"

print("Gemini client initialized!")

### Define a helper to extract a video ID from common YouTube URL formats.


In [ ]:
def extract_video_id(url):

    patterns = [
        r"(?:v=)([^&]+)",
        r"youtu\.be/([^?&]+)",
        r"shorts/([^?&]+)"        
    ]

    for pattern in patterns:
        match = re.search(pattern,url)

        if match:
            return match.group(1)

    raise ValueError("Invalid YouTube url")

### Choose a YouTube video URL and extract its video ID.


In [ ]:
url = "https://www.youtube.com/watch?v=Lx5QwuqVbg0"

video_id = extract_video_id(url)

print(video_id)

### Define a helper to fetch and combine the video transcript.


In [ ]:
def fetch_transcript(url):

    transcript_list = YouTubeTranscriptApi().fetch(video_id)
    transcript = []

    for snippet in transcript_list:
        transcript.append(snippet.text)

    return " ".join(transcript)

### Fetch the selected video transcript and preview its opening text.


In [ ]:
transcript = fetch_transcript(video_id)
print(transcript[:4000])

### Count the transcript characters, words, and tokens.


In [ ]:
encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode(transcript)

print("Characters:", len(transcript))
print("Words:", len(transcript.split()))
print("Tokens:", len(tokens))

### Define a helper to split the transcript into token-limited chunks.


In [ ]:
def chunk_transcript(text, max_tokens=2000):
    tokens = encoding.encode(text)

    chunks = []

    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i + max_tokens]

        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)

    return chunks

### Split the transcript into chunks and inspect each chunk’s token count.


In [ ]:
chunks = chunk_transcript(transcript)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(encoding.encode(chunk))} tokens")

### Visualize the token count for each transcript chunk.


In [ ]:
import matplotlib.pyplot as plt

chunk_sizes = [
    len(encoding.encode(chunk))
    for chunk in chunks
]

plt.figure(figsize=(10, 4))
plt.bar(range(1, len(chunk_sizes) + 1), chunk_sizes)

plt.title("Transcript Token Distribution")
plt.xlabel("Chunk Number")
plt.ylabel("Token Count")
plt.grid(axis="y", alpha=0.3)
plt.show()

### Define the Gemini prompt and helper for summarizing one transcript chunk.


In [ ]:
def summarize_chunk(chunk):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """
                You are an expert educational assistant.
                Summarize the provided video transcript.

                Focus on:
                - Main ideas
                - Important concepts
                - Key explanations
                - Practical examples

                Keep the summary concise but informative.
                Do not invent information.
                """
            },
            {
                "role": "user",
                "content": chunk
            }
        ]
    )

    return response.choices[0].message.content

### Summarize every transcript chunk and collect the results.


In [ ]:
chunk_summaries = []

for i, chunk in enumerate(chunks):
    print(f"Summarizing chunk {i+1}/{len(chunks)}...")

    summary = summarize_chunk(chunk)
    chunk_summaries.append(summary)

print("All chunks are summarized")

### Define a helper to combine chunk summaries into one organized summary.


In [ ]:
def generate_final_summary(summaries):

    combined_summaries = "\n\n".join(summaries)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """
                You are an expert educational assistant.

                Combine the provided summaries into one
                comprehensive and well-structured summary.

                Remove repetition.
                Preserve important concepts.
                Organize information using headings and bullet points.
                Do not introduce unsupported information.
                """
            },
            {
                "role": "user",
                "content": combined_summaries
            }
        ]
    )

    return response.choices[0].message.content

### Generate and display the final video summary.


In [ ]:
final_summary = generate_final_summary(chunk_summaries)

display(Markdown(final_summary))

### Set the instructions for turning the summary into detailed study notes.


In [ ]:
notes_system_prompt = """
You are an expert educational note-taking assistant.

Create detailed study notes from the provided video summary.

Organize the notes into:

1. Main Topic
2. Important Concepts
3. Detailed Explanations
4. Examples
5. Key Takeaways

Instructions:
- Use clear and simple language.
- Use headings and bullet points.
- Preserve important technical terminology.
- Include examples when available.
- Do not invent information.
- Avoid unnecessary repetition.

Return the notes in Markdown.
"""

### Define a helper that asks Gemini to generate study notes.


In [ ]:
def generate_study_notes(summary):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": notes_system_prompt
            },
            {
                "role": "user",
                "content": f"Create study notes from:\n\n{summary}"
            }
        ]
    )
    return response.choices[0].message.content

### Generate and display study notes from the final summary.


In [ ]:
study_notes = generate_study_notes(final_summary)
display(Markdown(study_notes))

### Set the instructions and JSON format for extracting key concepts.


In [ ]:
concepts_prompt = """
You are an expert technical educator.

Extract the most important concepts from the provided content.

For each concept, provide:
- Concept name
- Simple explanation
- Why it matters
- Example, if available

Return valid JSON in this format:

{
    "concepts": [
        {
            "name": "Concept name",
            "explanation": "Simple explanation",
            "importance": "Why it matters",
            "example": "Example"
        }
    ]
}

Do not invent information.
"""

### Define a helper to extract concepts as structured JSON.


In [ ]:
def extract_concepts(summary):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": concepts_prompt
            },
            {
                "role": "user",
                "content": summary
            }
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

### Extract concepts from the final summary and display each one.


In [ ]:
concepts = extract_concepts(final_summary)

for concept in concepts["concepts"]:

    display(Markdown(f"""
### {concept['name']}

**Explanation:** {concept['explanation']}

**Why it matters:** {concept['importance']}

**Example:** {concept['example']}

---
"""))

### Set the instructions and JSON format for generating flashcards from concepts.


In [ ]:
flashcard_prompt = """
You are an expert learning assistant.

Generate flashcards from the provided study concepts.

Requirements:
1. Generate 2-3 flashcards per important concept.
2. Questions should test understanding, not just memorization.
3. Answers should be concise and accurate.
4. Avoid duplicate questions.
5. Do not introduce information that isn't supported by the material.

Return valid JSON in this format:

{
    "flashcards": [
        {
            "question": "What is RAG?",
            "answer": "Retrieval-Augmented Generation..."
        }
    ]
}
"""

### Define a helper to generate flashcards as structured JSON.


In [ ]:
def generate_flashcards(concepts):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": flashcard_prompt
            },
            {
                "role": "user",
                "content": json.dumps(concepts)
            }
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

### Generate the flashcards and report how many were created.


In [ ]:
flashcards = generate_flashcards(concepts)

print("Total flashcards:", len(flashcards["flashcards"]))

### Build an interactive flashcard viewer with previous, flip, and next controls.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

cards = flashcards["flashcards"]

current_index = 0
show_answer = False

card_output = widgets.Output()

previous_button = widgets.Button(
    description="Previous",
    icon="arrow-left"
)

flip_button = widgets.Button(
    description="Show Answer",
    button_style="primary"
)

next_button = widgets.Button(
    description="Next",
    icon="arrow-right"
)

def render_card():

    global show_answer

    with card_output:
        card_output.clear_output(wait=True)

        card = cards[current_index]

        content = (
            card["answer"]
            if show_answer
            else card["question"]
        )

        heading = "Answer" if show_answer else "Question"

        display(HTML(f"""
        <div style="
            background:#111827;
            color:white;
            padding:30px;
            border-radius:16px;
            min-height:180px;
            text-align:center;
            font-family:sans-serif;
        ">
            <p style="color:#9CA3AF;">
                {heading} · Card {current_index + 1}/{len(cards)}
            </p>
            <h2>{content}</h2>
        </div>
        """))

    flip_button.description = (
        "Show Question" if show_answer else "Show Answer"
    )

def next_card(_):

    global current_index, show_answer

    current_index = (current_index + 1) % len(cards)
    show_answer = False

    render_card()

def previous_card(_):

    global current_index, show_answer

    current_index = (current_index - 1) % len(cards)
    show_answer = False

    render_card()

def flip_card(_):

    global show_answer

    show_answer = not show_answer

    render_card()

next_button.on_click(next_card)
previous_button.on_click(previous_card)
flip_button.on_click(flip_card)

render_card()

display(card_output)

display(widgets.HBox([
    previous_button,
    flip_button,
    next_button
]))

### Set the instructions and JSON format for creating Mcq's from the study notes.


In [ ]:
quiz_prompt = """
You are an expert educational quiz generator.

Generate 10 multiple-choice questions from the provided study notes.

Requirements:
1. Each question must have four options.
2. Only one option must be correct.
3. Include the correct answer.
4. Include a short explanation.
5. Cover different concepts.
6. Avoid duplicate questions.
7. Only use information supported by the notes.

Return valid JSON:

{
    "questions": [
        {
            "question": "What is RAG?",
            "options": {
                "A": "Retrieval-Augmented Generation",
                "B": "Random Answer Generation",
                "C": "Recursive Agent Graph",
                "D": "Retrieval Analysis Gateway"
            },
            "correct_answer": "A",
            "explanation": "RAG retrieves relevant information..."
        }
    ]
}
"""

### Define a helper that generates Mcq's from the study notes.


In [ ]:
def generate_quiz(study_notes):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": quiz_prompt
            },
            {
                "role": "user",
                "content": study_notes
            }
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

### Generate the quiz and report the number of questions.


In [ ]:
quiz = generate_quiz(study_notes)

print("Total questions:", len(quiz["questions"]))

### Answer Mcq's with interactive controls and track submitted answers.


In [75]:

import ipywidgets as widgets
from IPython.display import display, HTML

questions = quiz["questions"]

current_question = 0
score = 0
answers = {}
submitted = set()

question_area = widgets.VBox()

options_widget = widgets.RadioButtons(
    options=[],
    layout=widgets.Layout(width="100%")
)

feedback = widgets.HTML()

submit_button = widgets.Button(
    description="Submit Answer",
    button_style="primary"
)

previous_button = widgets.Button(
    description="Previous",
    button_style="info",
    icon="arrow-left",
    disabled=True
)

next_button = widgets.Button(
    description="Next Question",
    button_style="success",
    icon="arrow-right"
)


def render_question():

    q = questions[current_question]

    question_title = widgets.HTML(f"""
        <h3>Question {current_question + 1} / {len(questions)}</h3>
        <h2>{q['question']}</h2>
        <p>Select your answer:</p>
    """)

    options_widget.options = [
        (f"{key}: {value}", key)
        for key, value in q["options"].items()
    ]

    # Restore previously selected answer
    options_widget.value = answers.get(current_question)

    # Lock options if already submitted
    options_widget.disabled = current_question in submitted

    submit_button.disabled = current_question in submitted

    # Previous active only after first question
    previous_button.disabled = current_question == 0

    # Next remains enabled while questions remain
    next_button.disabled = False
    next_button.description = (
        "Finish Quiz"
        if current_question == len(questions) - 1
        else "Next Question"
    )


    if current_question in submitted:

        correct = questions[current_question]["correct_answer"]
        selected = answers[current_question]

        result = (
            "<h3 style='color:green'>Correct! 🎉</h3>"
            if selected == correct
            else "<h3 style='color:red'>Incorrect!</h3>"
        )

        feedback.value = f"""
            {result}
            <p><b>Correct answer:</b> {correct}</p>
            <p><b>Explanation:</b> {q['explanation']}</p>
        """

    else:
        feedback.value = ""

    question_area.children = [
        question_title,
        options_widget,
        submit_button,
        feedback,
        widgets.HBox([
            previous_button,
            next_button
        ])
    ]


def submit_answer(button):

    global score

    if options_widget.value is None:
        feedback.value = (
            "<p style='color:orange'>Select an answer first!</p>"
        )
        return

    selected = options_widget.value
    q = questions[current_question]

    correct = q["correct_answer"]

    answers[current_question] = selected
    submitted.add(current_question)

    if selected == correct:
        score += 1
        result = "<h3 style='color:green'>Correct! 🎉</h3>"
    else:
        result = "<h3 style='color:red'>Incorrect!</h3>"

    feedback.value = f"""
        {result}
        <p><b>Correct answer:</b> {correct}</p>
        <p><b>Explanation:</b> {q['explanation']}</p>
    """

    options_widget.disabled = True
    submit_button.disabled = True


def go_to_next(button):

    global current_question

    if current_question < len(questions) - 1:
        current_question += 1
        render_question()
    else:
        show_results()


def go_to_previous(button):

    global current_question

    if current_question > 0:
        current_question -= 1
        render_question()


def show_results():

    total = len(questions)
    percentage = (score / total) * 100

    question_area.children = [
        widgets.HTML(f"""
        <div style="
            background:#111827;
            color:white;
            padding:30px;
            border-radius:15px;
            text-align:center;
        ">
            <h2>Quiz Completed! 🎉</h2>
            <h1>{score}/{total}</h1>
            <h2>{percentage:.1f}%</h2>
            <p>You've completed your learning session.</p>
        </div>
        """)
    ]


submit_button.on_click(submit_answer)
next_button.on_click(go_to_next)
previous_button.on_click(go_to_previous)


render_question()
display(question_area)


### Review the answers, correct answers, and results in a summary table.


In [76]:
performance = pd.DataFrame([
    {
        "Question": i + 1,
        "Your Answer": answers.get(i, "Not answered"),
        "Correct Answer": q["correct_answer"],
        "Result": (
            "Correct"
            if answers.get(i) == q["correct_answer"]
            else "Incorrect"
        )
    }
    for i, q in enumerate(questions)
])

display(performance)

,Question,Your Answer,Correct Answer,Result
0,1,Not answered,B,Incorrect
1,2,Not answered,B,Incorrect
2,3,Not answered,C,Incorrect
3,4,Not answered,C,Incorrect
4,5,Not answered,A,Incorrect
5,6,Not answered,C,Incorrect
6,7,Not answered,B,Incorrect
7,8,Not answered,A,Incorrect
8,9,Not answered,B,Incorrect
9,10,Not answered,A,Incorrect


In [79]:
OUTPUT_DIR = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output directory is ready!")

Output directory is ready!
